# 05 - TF-IDF para clusterizacao baseline

Este notebook prepara a representacao textual da base processada filtrada para a primeira clusterizacao. A unidade de analise continua sendo uma descricao normalizada unica, com a frequencia preservada como peso interpretativo e para uso posterior em `sample_weight`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nofis_classifier.paths import find_project_root
from nofis_classifier.tfidf import (
    default_tfidf_configs,
    load_modeling_dataset,
    prepare_modeling_dataframe,
    run_tfidf_experiments,
    save_tfidf_reports,
)

pd.set_option("display.max_colwidth", 120)

In [2]:
PROJECT_ROOT = find_project_root()
DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "items-notas-fiscais-filtros-combinados" / "items-notas-fiscais-filtros-combinados-202501-202504.parquet"
REPORT_DIR = PROJECT_ROOT / "reports" / "tfidf"

DATASET_PATH

WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/items-notas-fiscais-filtros-combinados/items-notas-fiscais-filtros-combinados-202501-202504.parquet')

## Carregar e validar base

A base filtrada ja deve estar consolidada por `descricao_normalizada`. Este passo valida as colunas obrigatorias, remove apenas casos invalidos e garante que a frequencia total nao mudou.

In [3]:
raw_items = load_modeling_dataset(DATASET_PATH)
modeling_items = prepare_modeling_dataframe(raw_items)

assert modeling_items["descricao_normalizada"].is_unique
assert modeling_items["descricao_normalizada"].str.strip().ne("").all()
assert raw_items["frequencia"].sum() == modeling_items["frequencia"].sum()

display(modeling_items.head())
display(
    pd.DataFrame(
        {
            "descricoes_unicas": [len(modeling_items)],
            "frequencia_total": [int(modeling_items["frequencia"].sum())],
            "frequencia_media": [modeling_items["frequencia"].mean()],
            "primeiro_mes": [raw_items["primeiro_mes"].min()],
            "ultimo_mes": [raw_items["ultimo_mes"].max()],
        }
    )
)

,descricao_id,descricao_normalizada,frequencia,descricao_original_exemplo,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,0,000 alho branco kg,1,000-ALHO BRANCO KG,7032090.0,"Alhos, frescos ou refrigerados, exceto para semeadura"
1,1,000 batata especial kg,1,000-BATATA ESPECIAL KG,7019000.0,"Batatas, frescas ou refrigeradas, exceto para semeadura"
2,2,000 cebola conserva kg,1,000-CEBOLA CONSERVA KG,7031019.0,"Cebolas, frescas ou refrigeradas, exceto para semeadura"
3,3,000 cenoura kg,1,000-CENOURA KG,7061000.0,"Cenouras e nabos, frescos ou refrigerados"
4,4,000 kiwi importado kg,1,000-KIWI IMPORTADO KG,8105000.0,"Kiwis (quivis), frescos"


,descricoes_unicas,frequencia_total,frequencia_media,primeiro_mes,ultimo_mes
0,27800,134624,4.84259,2025-01,2025-04


## Comparar configuracoes de TF-IDF

As configuracoes comparam palavras, palavras com bigramas e caracteres `char_wb`. Para descricoes fiscais, a configuracao de caracteres tende a lidar melhor com abreviacoes, marcas, erros de digitacao e variacoes curtas.

In [4]:
configs = default_tfidf_configs()
summary, vocabulary_samples, vectorizers, matrices = run_tfidf_experiments(modeling_items, configs)

for config in configs:
    matrix = matrices[config.name]
    assert matrix.shape[0] == len(modeling_items)
    assert matrix.shape[1] > 0
    assert matrix.shape[1] <= config.max_features

summary.sort_values(["analyzer", "min_df", "feature_count"])

,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
2,char_wb_3_5_min_df_2,char_wb,"(3, 5)",2,30000,0.95,True,l2,27800,30000,1615099,834000000,0.001937,0.998063
5,char_wb_3_5_min_df_5,char_wb,"(3, 5)",5,30000,0.95,True,l2,27800,23507,1592518,653494600,0.002437,0.997563
0,word_unigram_min_df_2,word,"(1, 1)",2,10000,0.95,True,l2,27800,6676,128897,185592800,0.000695,0.999305
1,word_unigram_bigram_min_df_2,word,"(1, 2)",2,10000,0.95,True,l2,27800,10000,173830,278000000,0.000625,0.999375
3,word_unigram_min_df_5,word,"(1, 1)",5,10000,0.95,True,l2,27800,3231,119866,89821800,0.001334,0.998666
4,word_unigram_bigram_min_df_5,word,"(1, 2)",5,10000,0.95,True,l2,27800,6469,161352,179838200,0.000897,0.999103


In [5]:
baseline_config_name = "char_wb_3_5_min_df_2"
baseline_vectorizer = vectorizers[baseline_config_name]
baseline_matrix = matrices[baseline_config_name]

display(summary[summary["name"] == baseline_config_name])
display(vocabulary_samples[vocabulary_samples["config_name"] == baseline_config_name].head(40))

baseline_matrix.shape

,name,analyzer,ngram_range,min_df,max_features,max_df,sublinear_tf,norm,document_count,feature_count,non_zero_values,total_cells,density,sparsity
2,char_wb_3_5_min_df_2,char_wb,"(3, 5)",2,30000,0.95,True,l2,27800,30000,1615099,834000000,0.001937,0.998063


,config_name,sample_type,description,frequency,rank,term,value
115,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,1,0,NaN
116,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,2,00,NaN
117,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,3,00,NaN
118,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,4,000,NaN
119,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,5,000,NaN
120,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,6,0000,NaN
121,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,7,0001,NaN
122,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,8,0002,NaN
123,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,9,0005,NaN
124,char_wb_3_5_min_df_2,vocabulary_sample,NaN,NaN,10,000x,NaN


(27800, 30000)

## Salvar relatorios

Os CSVs abaixo registram a comparacao das representacoes e amostras do vocabulario para apoiar a escrita da metodologia e a escolha da configuracao baseline.

In [6]:
save_tfidf_reports(summary, vocabulary_samples, REPORT_DIR)

sorted(path.relative_to(PROJECT_ROOT) for path in REPORT_DIR.glob("*.csv"))

[WindowsPath('reports/tfidf/tfidf_experiments_summary.csv'),
 WindowsPath('reports/tfidf/tfidf_vocabulary_samples.csv')]